# Dice Roller - Design Notes

Purpose: A CLI program to generate random numbers for TTRPG games. User should stipulate the 'facets' of the dice, and the quantity of dice for each roll. For example: `roll(20, 1)` or `roll(6, 3)` Store the rolls in a CSV file. And generate a historical graph of numbers rolled per dice type over specified time period expressed in days.

## Imports

In [23]:
import random
import json
import argparse
from datetime import datetime
from pathlib import Path


## Constants

In [24]:
DICE_ROLL_DIR = Path.home() / '.dice_roll'
ROLLS_FILE = DICE_ROLL_DIR / 'rolls.json'


## Functions

In [ ]:
def get_current_session(gap_hours=4):
    """
    Determine current session ID.
    If no rolls exist, start session-001.
    If last roll was > gap_hours ago, increment session.
    Otherwise continue current session.
    """
    
    # No file yet, first session
    if not ROLLS_FILE.exists():
        return "session-001"
    
    with open(ROLLS_FILE, 'r') as f:
        data = json.load(f)
    
    # File exists but is empty
    if not data:
        return "session-001"
    
    # Check last roll's timestamp
    last_roll = data[-1]
    last_timestamp = datetime.fromisoformat(last_roll['timestamp'])
    time_since = datetime.now() - last_timestamp
    
    last_session = last_roll['session_id']
    session_num = int(last_session.split('-')[1])
    
    if time_since.total_seconds() > (gap_hours * 3600):
        # New session
        return f"session-{(session_num + 1):03d}"
    else:
        # Continue current session
        return last_session

def roll(dice_type, quantity, drop=None, gap_hours=4):
    """
    Main roll function - rolls dice, prints result, logs to JSON.
    dice_type: int - number of faces (20, 6, 12, etc.)
    quantity: int - number of dice to roll
    gap_hours: int - hours before a new session is declared
    """
    
    session_id = get_current_session(gap_hours)
    results = [random.randint(1, dice_type) for _ in range(quantity)]
    all_rolls = results.copy() # snapshot of all rolls before dropping
    
    if drop == 'low':
        dropped = min(results)
        results.remove(dropped)
    elif drop == 'high':
        dropped = max(results)
        results.remove(dropped)
    else:
        dropped = None
    kept = results if drop else None
    total = sum(results)
    
    # Print to stdout
    print(f"Session : {session_id}")
    print(f"Roll    : {quantity}d{dice_type}" + (f" (drop {drop})" if drop else ""))
    print(f"Results : {all_rolls}")
    if drop:
        print(f"Dropped : {dropped}")
        print(f"Kept    : {kept}")
    print(f"Total   : {total}")
    
    # Log to JSON
    save_roll(dice_type, quantity, all_rolls, total, session_id, drop=drop, dropped=dropped, kept=kept)
    
    return results, total

def save_roll(dice_type, quantity, individual_rolls, total, session_id, drop=None, dropped=None, kept=None):
    """Append a roll record to rolls.json"""
    
    record = {
        "timestamp": datetime.now().isoformat(),
        "session_id": session_id,
        "dice_type": f"d{dice_type}",
        "quantity": quantity,
        "rolls": individual_rolls,
        "drop": drop,
        "dropped": dropped,
        "kept": kept,
        "total": total
    }
    
    # Load existing data if file exists, otherwise start fresh
    if ROLLS_FILE.exists():
        with open(ROLLS_FILE, 'r') as f:
            data = json.load(f)
    else:
        data = []
    
    # Append new record
    data.append(record)
    
    # Write back
    with open(ROLLS_FILE, 'w') as f:
        json.dump(data, f, indent=2)
        
    ## FOR DEBUGGING PURPOSES, UNCOMMENT THE FOLLOWING LINE TO SEE LOGGED DATA
    # print(f"Logged: {session_id} | d{dice_type} x{quantity} | {individual_rolls} | Total: {total}")

def show_history(num_sessions=1):
    """
    Display roll history for the last N sessions.
    num_sessions: int - how many sessions to display
    """
    
    if not ROLLS_FILE.exists():
        print("No roll history found.")
        return
    
    with open(ROLLS_FILE, 'r') as f:
        data = json.load(f)
    
    if not data:
        print("No rolls recorded yet.")
        return
    
    # Get unique session IDs preserving order
    seen = []
    for roll in data:
        if roll['session_id'] not in seen:
            seen.append(roll['session_id'])
    
    # Grab the last N sessions
    sessions_to_show = seen[-num_sessions:]
    
    # Filter and display
    for session in sessions_to_show:
        print(f"\n{'='*40}")
        print(f"  {session}")
        print(f"{'='*40}")
        session_rolls = [r for r in data if r['session_id'] == session]
        for r in session_rolls:
            print(f"  {r['dice_type']:>4} x{r['quantity']} | {r['rolls']} | Total: {r['total']}")

def main():
    parser = argparse.ArgumentParser(
        description='🎲 TTRPG Die Roller',
        epilog='Example: dice_roll 20 1 or dice_roll 6 3 --gap 6'
    )
    
    # Positional arguments
    parser.add_argument('dice_type', 
                        nargs='?',          # optional
                        type=int, 
                        help='Number of faces on the die (e.g. 20, 6, 12)')
    
    parser.add_argument('quantity', 
                        nargs='?',          # optional
                        type=int,
                        default=1,
                        help='Number of dice to roll (default: 1)')
    
    # Optional flags
    parser.add_argument('--gap',
                        type=int,
                        default=4,
                        help='Hours between sessions (default: 4)')
    
    parser.add_argument('--history',
                        type=int,
                        metavar='N',
                        help='Show last N sessions')
    
    args = parser.parse_args()
    
    # Route to the right function
    if args.history:
        show_history(args.history)
    elif args.dice_type:
        roll(args.dice_type, args.quantity, args.gap)
    else:
        parser.print_help()

if __name__ == "__main__":
    main()

Session : session-001
Roll    : 1d20 (drop 4)
Results : [1]
Dropped : None
Kept    : [1]
Total   : 1


## Testing

In [ ]:
# Test History Display
show_history(1)

# Test Rolls
roll(6, 4, drop='low')    # character creation
roll(20, 2, drop='low')   # advantage
roll(20, 2, drop='high')  # disadvantage
roll(20, 1)               # standard, no drop fields

#check the rolls.json file
print(ROLLS_FILE.read_text())

# Test Session Detection
session = get_current_session()
print(f"Current session: {session}")

# Retest History Display
show_history(1)

